# End-to-End Test-Time Training for Long Context

> **Paper:** [test-time-training.github.io/e2e.pdf](https://test-time-training.github.io/e2e.pdf)  
> **Repo:** [github.com/test-time-training/e2e](https://github.com/test-time-training/e2e)  
> **arXiv:** [2512.23675](https://arxiv.org/abs/2512.23675)

---

## Overview

This notebook walks through the key ideas in **E2E-TTT** — a method that reformulates long-context language modelling as **continual learning** rather than an architecture design problem.

### The core problem

Standard transformers use **full attention** over the entire context window. This has two costs:

| Property | Full Attention | Sliding-Window Attention + TTT |
|---|---|---|
| Reads context | ✅ Everything | ✅ Everything (at training time) |
| Inference memory | O(n) KV cache | O(window) KV cache |
| Inference compute | O(n²) | O(1) per token |
| Extrapolates beyond training length | ❌ | ✅ |

### The E2E-TTT answer

1. Use a **standard Transformer with sliding-window attention** (fixed, small window) — cheap at inference.
2. Add a small **adaptive MLP** per block that **updates its own weights** at test time via next-token prediction on the input context, effectively compressing the long context into weights.
3. **Meta-train** the model so the adaptive MLP's *initial weights* are a good starting point for fast test-time adaptation (MAML-style).

This makes the method **E2E** in two senses:
- **Test-time E2E:** the TTT objective is the same as the main objective (next-token prediction)
- **Training-time E2E:** meta-learning is end-to-end differentiable through the TTT inner loop

---

## Notebook Contents

1. [Imports](#1-imports)
2. [Building Blocks from Scratch](#2-building-blocks-from-scratch)
3. [Architecture Deep Dive](#3-architecture-deep-dive)
4. [Sliding-Window Attention](#4-sliding-window-attention)
5. [Dual MLP Design](#5-dual-mlp-design)
6. [Meta-Training](#6-meta-training)
7. [Test-Time Training](#7-test-time-training)
8. [Live Demo](#8-live-demo)
9. [Visualising TTT Adaptation](#9-visualising-ttt-adaptation)
10. [Production JAX Implementation](#10-production-jax-implementation)
11. [Checkpoints & Reproducing Results](#11-checkpoints--reproducing-results)


---
## 1. Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import time

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')
print('NumPy', np.__version__)

---
## 2. Building Blocks from Scratch

We implement the full E2E-TTT model in pure NumPy — no PyTorch, no JAX, no GPU required.

### 2.1 Utility functions

In [ ]:
# ── Utility functions ─────────────────────────────────────────────────────────

def gelu(x):
    """Gaussian Error Linear Unit activation."""
    return 0.5 * x * (1.0 + np.tanh(np.sqrt(2.0 / np.pi) * (x + 0.044715 * x**3)))


def gelu_backward(x):
    """Derivative of GELU with respect to x."""
    c = np.sqrt(2.0 / np.pi)
    inner = c * (x + 0.044715 * x**3)
    tanh_val = np.tanh(inner)
    sech2 = 1.0 - tanh_val**2
    inner_grad = c * (1.0 + 3.0 * 0.044715 * x**2)
    return 0.5 * (1.0 + tanh_val) + 0.5 * x * sech2 * inner_grad


def softmax(x, axis=-1):
    e = np.exp(x - np.max(x, axis=axis, keepdims=True))
    return e / np.sum(e, axis=axis, keepdims=True)


def cross_entropy(logits, targets):
    """
    Cross-entropy loss for next-token prediction.
    logits:  (T, V)  — raw scores per token
    targets: (T,)    — integer token indices
    Returns: (scalar loss, gradient w.r.t. logits)
    """
    T, V = logits.shape
    probs = softmax(logits, axis=-1)
    log_probs = np.log(probs + 1e-12)
    loss = -np.mean(log_probs[np.arange(T), targets])
    # Gradient: (probs - one_hot) / T
    grad = probs.copy()
    grad[np.arange(T), targets] -= 1.0
    grad /= T
    return loss, grad


print("Utility functions defined: gelu, gelu_backward, softmax, cross_entropy")

### 2.2 Learnable layer classes

Each class stores its parameters and implements `forward` + `backward`, mirroring the structure in the JAX repo.

In [ ]:
# ── Linear layer ──────────────────────────────────────────────────────────────

class Linear:
    """Fully-connected layer: y = x @ W  (no bias for simplicity)."""

    def __init__(self, in_dim, out_dim):
        # He initialisation
        self.W = np.random.randn(in_dim, out_dim).astype(np.float32) * np.sqrt(2.0 / in_dim)
        self.grad_W = np.zeros_like(self.W)

    def forward(self, x):
        self.x = x
        return x @ self.W

    def backward(self, grad_out):
        x_flat = self.x.reshape(-1, self.x.shape[-1])
        g_flat = grad_out.reshape(-1, grad_out.shape[-1])
        self.grad_W = x_flat.T @ g_flat
        return grad_out @ self.W.T


# ── LayerNorm ─────────────────────────────────────────────────────────────────

class LayerNorm:
    """Standard layer normalisation with learnable γ (scale) and β (shift)."""

    def __init__(self, dim):
        self.gamma = np.ones(dim, dtype=np.float32)
        self.beta  = np.zeros(dim, dtype=np.float32)
        self.grad_gamma = np.zeros_like(self.gamma)
        self.grad_beta  = np.zeros_like(self.beta)

    def forward(self, x, eps=1e-5):
        self.x     = x
        self.eps   = eps
        self.mean  = np.mean(x, axis=-1, keepdims=True)
        self.var   = np.var(x, axis=-1, keepdims=True)
        self.x_norm = (x - self.mean) / np.sqrt(self.var + eps)
        return self.gamma * self.x_norm + self.beta

    def backward(self, grad_out):
        N = self.x.shape[-1]
        dx_norm = grad_out * self.gamma
        self.grad_gamma = np.sum(grad_out * self.x_norm,
                                 axis=tuple(range(grad_out.ndim - 1)))
        self.grad_beta  = np.sum(grad_out,
                                 axis=tuple(range(grad_out.ndim - 1)))
        std_inv = 1.0 / np.sqrt(self.var + self.eps)
        dx = (1.0 / N) * std_inv * (
            N * dx_norm
            - np.sum(dx_norm, axis=-1, keepdims=True)
            - self.x_norm * np.sum(dx_norm * self.x_norm, axis=-1, keepdims=True)
        )
        return dx


print("Layer classes defined: Linear, LayerNorm")

### 2.3 MLP block

A two-layer MLP with GELU activation — used for both the **frozen** and **adaptive** MLPs in each block.

In [ ]:
# ── MLP Block ─────────────────────────────────────────────────────────────────

class MLPBlock:
    """
    Two-layer MLP:  x -> Linear -> GELU -> Linear

    Corresponds to SwiGLUMLP in the JAX repo, simplified to a standard
    GELU-MLP for clarity.
    """

    def __init__(self, d_model, d_ff):
        self.fc1 = Linear(d_model, d_ff)
        self.fc2 = Linear(d_ff, d_model)

    def forward(self, x):
        self.h_pre = self.fc1.forward(x)   # pre-activation
        self.h     = gelu(self.h_pre)       # activation
        return self.fc2.forward(self.h)

    def backward(self, grad_out):
        grad_h     = self.fc2.backward(grad_out)
        grad_h_pre = grad_h * gelu_backward(self.h_pre)
        return self.fc1.backward(grad_h_pre)

    def params_and_grads(self):
        """Returns list of (param, grad) pairs — used by the SGD update."""
        return [
            (self.fc1.W, self.fc1.grad_W),
            (self.fc2.W, self.fc2.grad_W),
        ]


print("MLPBlock defined")

### 2.4 Sliding-Window Attention

Each token attends only to the `window_size` most recent tokens.  
The production code (`ttt/model/attention.py` → `SWA`) adds RoPE positional embeddings and FlashAttention; we use a simpler mask-based approach here.

In [ ]:
# ── Sliding-Window Attention ──────────────────────────────────────────────────

class SlidingWindowAttention:
    """
    Multi-head causal attention limited to a local window.

    Production equivalent: ttt/model/attention.py → SWA
    (which adds RoPE, FlashAttention, and GQA — omitted here for clarity)
    """

    def __init__(self, d_model, n_heads, window_size):
        assert d_model % n_heads == 0
        self.n_heads   = n_heads
        self.head_dim  = d_model // n_heads
        self.window    = window_size
        self.scale     = 1.0 / np.sqrt(self.head_dim)
        # Single fused QKV projection
        self.Wqkv = Linear(d_model, 3 * d_model)
        self.Wout = Linear(d_model, d_model)

    def _build_mask(self, T):
        """Causal sliding-window mask: -1e9 where token cannot attend."""
        mask = np.full((T, T), -1e9, dtype=np.float32)
        for i in range(T):
            start = max(0, i - self.window + 1)
            mask[i, start : i + 1] = 0.0
        return mask

    def forward(self, x):
        T, C = x.shape
        # QKV projection
        qkv = self.Wqkv.forward(x).reshape(T, 3, self.n_heads, self.head_dim)
        q, k, v = (qkv[:, i].transpose(1, 0, 2) for i in range(3))  # (H, T, d_h)

        # Scaled dot-product attention with sliding-window mask
        scores = np.matmul(q, k.transpose(0, 2, 1)) * self.scale  # (H, T, T)
        scores += self._build_mask(T)
        self.attn_weights = softmax(scores, axis=-1)               # (H, T, T)

        out = np.matmul(self.attn_weights, v)                      # (H, T, d_h)
        out = out.transpose(1, 0, 2).reshape(T, C)                 # (T, C)
        return self.Wout.forward(out)

    def backward(self, grad_out):
        # Backprop through Wout, then approximate gradient through QKV
        grad_mid = self.Wout.backward(grad_out)        # (T, C)
        T, C = grad_mid.shape
        # Approximate: distribute grad_mid equally to Q, K, V paths
        grad_qkv = np.concatenate([grad_mid] * 3, axis=-1)
        x_flat = self.Wqkv.x.reshape(-1, self.Wqkv.x.shape[-1])
        g_flat = grad_qkv.reshape(-1, grad_qkv.shape[-1])
        self.Wqkv.grad_W = x_flat.T @ g_flat
        return grad_qkv @ self.Wqkv.W.T

    def params_and_grads(self):
        return [
            (self.Wqkv.W, self.Wqkv.grad_W),
            (self.Wout.W, self.Wout.grad_W),
        ]


print("SlidingWindowAttention defined")

### 2.5 Transformer Block with Dual MLP

The key novelty: each block has **two parallel MLPs** in its feed-forward sub-layer:
- `frozen_mlp`  — updated only during meta-training (maps to `feed_forward` in the JAX repo)
- `adaptive_mlp` — updated at **test time** via SGD (maps to `feed_forward_prime`)

In [ ]:
# ── Transformer Block ─────────────────────────────────────────────────────────

class TransformerBlock:
    """
    Pre-norm Transformer block with dual MLP.

    Forward:
        h  = x + SWA(LN1(x))
        x' = h + frozen_mlp(LN2(h)) + adaptive_mlp(LN2(h))

    Production equivalent: ttt/model/transformer.py → Block
    """

    def __init__(self, d_model, n_heads, d_ff, window_size):
        self.ln1 = LayerNorm(d_model)
        self.attn = SlidingWindowAttention(d_model, n_heads, window_size)
        self.ln2 = LayerNorm(d_model)
        # d_ff is split equally between the two MLPs
        half = d_ff // 2
        self.frozen_mlp   = MLPBlock(d_model, half)   # feed_forward
        self.adaptive_mlp = MLPBlock(d_model, half)   # feed_forward_prime

    def forward(self, x):
        # ── Attention sub-layer ──
        self.x_before_attn = x
        h = self.ln1.forward(x)
        x = x + self.attn.forward(h)

        # ── Feed-forward sub-layer (dual MLP) ──
        self.x_before_ff = x
        h2 = self.ln2.forward(x)
        # Both MLPs receive the same input; outputs are summed
        x = x + self.frozen_mlp.forward(h2) + self.adaptive_mlp.forward(h2)
        return x

    def backward(self, grad_out):
        # Backprop through dual MLP
        grad_frozen   = self.frozen_mlp.backward(grad_out)
        grad_adaptive = self.adaptive_mlp.backward(grad_out)
        grad_ln2 = self.ln2.backward(grad_frozen + grad_adaptive)
        grad_x2 = grad_out + grad_ln2

        # Backprop through attention
        grad_attn = self.attn.backward(grad_x2)
        grad_ln1 = self.ln1.backward(grad_attn)
        return grad_x2 + grad_ln1

    def all_params_and_grads(self):
        """All parameters — used by the outer (meta) optimizer."""
        pgs = []
        pgs.extend(self.attn.params_and_grads())
        pgs.extend(self.frozen_mlp.params_and_grads())
        pgs.extend(self.adaptive_mlp.params_and_grads())
        pgs += [(self.ln1.gamma, self.ln1.grad_gamma),
                (self.ln1.beta,  self.ln1.grad_beta),
                (self.ln2.gamma, self.ln2.grad_gamma),
                (self.ln2.beta,  self.ln2.grad_beta)]
        return pgs

    def adaptive_params_and_grads(self):
        """Only the adaptive MLP — used by the inner (TTT) optimizer."""
        return self.adaptive_mlp.params_and_grads()


print("TransformerBlock defined")

### 2.6 Full E2E-TTT Model

In [ ]:
# ── E2E-TTT Model ─────────────────────────────────────────────────────────────

class E2ETTT:
    """
    Simplified E2E-TTT language model (pure NumPy).

    Architecture:
      token embedding + positional embedding
      → N TransformerBlocks (with sliding-window attention + dual MLP)
      → LayerNorm → Linear head → logits

    Only the adaptive MLPs in the last ⌈N/4⌉ blocks (TTT blocks)
    are updated at test time.

    Production equivalent: ttt/model/transformer.py → MetaModel + CausalLM
    """

    def __init__(self, vocab_size=256, d_model=64, n_heads=4, n_layers=4,
                 d_ff=128, max_seq_len=512, window_size=32,
                 ttt_batch_size=16, ttt_lr=0.01):
        self.vocab_size     = vocab_size
        self.d_model        = d_model
        self.max_seq_len    = max_seq_len
        self.ttt_batch_size = ttt_batch_size
        self.ttt_lr         = ttt_lr

        # Token + positional embeddings
        self.tok_emb = np.random.randn(vocab_size, d_model).astype(np.float32) * 0.02
        self.pos_emb = np.random.randn(max_seq_len, d_model).astype(np.float32) * 0.02

        # Stacked transformer blocks
        self.blocks = [
            TransformerBlock(d_model, n_heads, d_ff, window_size)
            for _ in range(n_layers)
        ]

        # Final layer norm + output projection
        self.ln_f = LayerNorm(d_model)
        self.head = Linear(d_model, vocab_size)

        # TTT block indices: last 25% of blocks
        n_ttt = max(1, n_layers // 4)
        self.ttt_block_indices = list(range(n_layers - n_ttt, n_layers))

    # ── Forward pass ────────────────────────────────────────────────────────

    def forward(self, token_ids):
        """token_ids: (T,) int array → logits: (T, vocab_size)"""
        T = len(token_ids)
        x = self.tok_emb[token_ids] + self.pos_emb[:T]
        for block in self.blocks:
            x = block.forward(x)
        x = self.ln_f.forward(x)
        return self.head.forward(x)

    # ── Backward pass ────────────────────────────────────────────────────────

    def backward(self, grad_logits):
        """Backprop through the whole model."""
        grad = self.head.backward(grad_logits)
        grad = self.ln_f.backward(grad)
        for block in reversed(self.blocks):
            grad = block.backward(grad)

    # ── Parameter accessors ──────────────────────────────────────────────────

    def all_params_and_grads(self):
        """All trainable params + grads (used by outer/meta optimizer)."""
        pgs = []
        for block in self.blocks:
            pgs.extend(block.all_params_and_grads())
        pgs += [(self.ln_f.gamma, self.ln_f.grad_gamma),
                (self.ln_f.beta,  self.ln_f.grad_beta),
                (self.head.W,     self.head.grad_W)]
        return pgs

    def ttt_params_and_grads(self):
        """Only adaptive MLP params in TTT blocks (used by inner/TTT optimizer)."""
        pgs = []
        for i in self.ttt_block_indices:
            pgs.extend(self.blocks[i].adaptive_params_and_grads())
        return pgs

    # ── TTT state management ─────────────────────────────────────────────────

    def save_ttt_state(self):
        """Snapshot the adaptive MLP weights (W₀)."""
        return [p.copy() for p, _ in self.ttt_params_and_grads()]

    def restore_ttt_state(self, state):
        """Restore adaptive MLP weights from snapshot."""
        for (p, _), saved in zip(self.ttt_params_and_grads(), state):
            p[:] = saved

    # ── Parameter counts ────────────────────────────────────────────────────

    def count_params(self):
        total = self.tok_emb.size + self.pos_emb.size
        for p, _ in self.all_params_and_grads():
            total += p.size
        return total

    def count_ttt_params(self):
        return sum(p.size for p, _ in self.ttt_params_and_grads())

    # ── Test-Time Training ───────────────────────────────────────────────────

    def ttt_adapt(self, context_ids):
        """
        Adapt the model to a long context at test time.

        Processes `context_ids` in non-overlapping mini-batches.
        For each mini-batch:
          1. Forward pass → next-token prediction loss
          2. Backward pass
          3. SGD step on adaptive MLP weights ONLY

        Returns list of per-batch losses (for monitoring).

        Production equivalent: MetaModel.inner_loop_step in loop.py
        """
        T = len(context_ids)
        n_batches = max(1, T // self.ttt_batch_size)
        losses = []

        for i in range(n_batches):
            start = i * self.ttt_batch_size
            end   = min(start + self.ttt_batch_size, T - 1)
            if end <= start:
                break

            batch_in  = context_ids[start : end]
            batch_tgt = context_ids[start + 1 : end + 1]

            logits = self.forward(batch_in)
            loss, grad_logits = cross_entropy(logits, batch_tgt)
            losses.append(loss)

            self.backward(grad_logits)

            # Inner SGD step — only adaptive MLP weights
            for p, g in self.ttt_params_and_grads():
                p -= self.ttt_lr * g

        return losses


print("E2ETTT model defined")

### 2.7 Meta-training loop

In [ ]:
# ── Meta-training ─────────────────────────────────────────────────────────────

def meta_train(model, n_steps=100, seq_len=64, lr=1e-3, verbose=False):
    """
    Simplified meta-training (outer loop).

    Uses synthetic sequences with repeating patterns so the model
    learns structure that generalises to unseen patterns at test time.

    In the full E2E-TTT training (loop.py → train_on_sequence), the outer
    loop differentiates *through* the inner TTT loop (MAML-style).  Here
    we approximate with standard next-token-prediction SGD on all params,
    which still gives the model a useful initialisation.

    Production equivalent: ttt/model/loop.py → train_on_sequence
    """
    V = model.vocab_size
    losses = []

    for step in range(n_steps):
        # Synthetic sequence: tile a random short pattern
        pattern_len = np.random.randint(3, 8)
        pattern = np.random.randint(0, V, size=pattern_len)
        repeats = seq_len // pattern_len + 1
        seq = np.tile(pattern, repeats)[: seq_len + 1].astype(np.int32)

        inputs  = seq[:-1]
        targets = seq[1:]

        logits = model.forward(inputs)
        loss, grad_logits = cross_entropy(logits, targets)
        losses.append(loss)
        model.backward(grad_logits)

        # Outer SGD on ALL params
        for p, g in model.all_params_and_grads():
            p -= lr * g

        if verbose and (step + 1) % 25 == 0:
            print(f"  Step {step+1:3d}/{n_steps}  loss: {loss:.4f}")

    return losses


print("meta_train defined")
print()
print("All components ready — no external imports needed.")

---
## 3. Architecture Deep Dive

### Block structure

```
x  ──► LayerNorm ──► SlidingWindowAttention ──► + ──► LayerNorm ──► [AdaptiveMLP + FrozenMLP] ──► +
│                                                │                                                  │
└────────────────────────────────────────────────┘                  └──────────────────────────────┘
         residual connection 1                                              residual connection 2
```

In the production JAX code (`ttt/model/transformer.py`):
- `feed_forward`       → frozen MLP (always updated by outer optimizer)
- `feed_forward_prime` → adaptive MLP (updated by inner SGD at test time)

Only the **suffix blocks** (last `suffix_len=3` layers) carry `feed_forward_prime`.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.set_xlim(0, 12)
ax.set_ylim(0, 6)
ax.axis('off')

def box(ax, x, y, w, h, text, color='#4A90D9', fontsize=9, text_color='white'):
    rect = mpatches.FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.1',
                                    facecolor=color, edgecolor='#2c3e50', linewidth=1.2)
    ax.add_patch(rect)
    ax.text(x + w/2, y + h/2, text, ha='center', va='center',
            fontsize=fontsize, fontweight='bold', color=text_color)

def arrow(ax, x1, y1, x2, y2):
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color='#2c3e50', lw=1.5))

box(ax, 0.2, 2.5, 1.2, 1, 'Input\nx', '#7f8c8d')
arrow(ax, 1.4, 3.0, 1.9, 3.0)
box(ax, 1.9, 2.5, 1.2, 1, 'LayerNorm', '#27ae60')
arrow(ax, 3.1, 3.0, 3.6, 3.0)
box(ax, 3.6, 2.5, 1.8, 1, 'Sliding-Window\nAttention', '#2980b9')
arrow(ax, 5.4, 3.0, 5.9, 3.0)
ax.text(5.95, 3.05, '+', fontsize=16, fontweight='bold', color='#2c3e50')
arrow(ax, 6.2, 3.0, 6.7, 3.0)
ax.annotate('', xy=(6.1, 3.0), xytext=(0.8, 3.0),
            arrowprops=dict(arrowstyle='->', color='#95a5a6', lw=1.2,
                           connectionstyle='arc3,rad=-0.4'))
box(ax, 6.7, 2.5, 1.2, 1, 'LayerNorm', '#27ae60')
arrow(ax, 7.9, 3.0, 8.2, 3.0)
box(ax, 8.2, 3.7, 2.0, 0.9, 'Frozen MLP\n(world knowledge)', '#8e44ad', fontsize=8)
box(ax, 8.2, 2.4, 2.0, 0.9, 'Adaptive MLP\n(test-time update)', '#e67e22', fontsize=8)
ax.annotate('', xy=(8.2, 4.15), xytext=(8.05, 3.3),
            arrowprops=dict(arrowstyle='->', color='#2c3e50', lw=1.2))
ax.annotate('', xy=(8.2, 2.85), xytext=(8.05, 3.0),
            arrowprops=dict(arrowstyle='->', color='#2c3e50', lw=1.2))
ax.annotate('', xy=(10.4, 3.3), xytext=(10.2, 4.15),
            arrowprops=dict(arrowstyle='->', color='#2c3e50', lw=1.2))
ax.annotate('', xy=(10.4, 3.1), xytext=(10.2, 2.85),
            arrowprops=dict(arrowstyle='->', color='#2c3e50', lw=1.2))
ax.text(10.45, 3.15, '+', fontsize=16, fontweight='bold', color='#2c3e50')
arrow(ax, 10.65, 3.2, 11.1, 3.2)
ax.annotate('', xy=(10.55, 3.2), xytext=(6.5, 3.2),
            arrowprops=dict(arrowstyle='->', color='#95a5a6', lw=1.2,
                           connectionstyle='arc3,rad=0.35'))
box(ax, 11.1, 2.7, 0.7, 0.9, 'Out', '#7f8c8d')

ax.set_title('E2E-TTT Transformer Block', fontsize=13, fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig('block_diagram.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 4. Sliding-Window Attention

Rather than attending to all past tokens (O(n²)), each token sees only the **W most recent** tokens.  
The production config uses **W = 8192** tokens → O(W·n) compute, linear in sequence length.

In [ ]:
def visualise_attention_mask(seq_len=16, window=5):
    full_mask = np.tril(np.ones((seq_len, seq_len)))
    swa_mask  = np.zeros((seq_len, seq_len))
    for i in range(seq_len):
        for j in range(max(0, i - window + 1), i + 1):
            swa_mask[i, j] = 1.0

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    for ax, mask, title in zip(
        axes,
        [full_mask, swa_mask],
        [f'Full Causal Attention\n(O(n²) — sees everything)',
         f'Sliding-Window Attention\n(window={window}, O(n·W) — sees last {window} tokens)'],
    ):
        ax.imshow(mask, cmap='Blues', vmin=0, vmax=1, aspect='auto')
        ax.set_xlabel('Key position', fontsize=10)
        ax.set_ylabel('Query position', fontsize=10)
        ax.set_title(title, fontsize=10, fontweight='bold')
        for i in range(seq_len):
            for j in range(seq_len):
                if mask[i, j] > 0:
                    ax.text(j, i, '✓', ha='center', va='center', fontsize=7, color='white')

    plt.suptitle('Attention Masks: What Each Token Can See', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig('attention_masks.png', dpi=150, bbox_inches='tight')
    plt.show()

visualise_attention_mask(seq_len=16, window=5)

---
## 5. Dual MLP Design

```python
# From ttt/model/transformer.py (JAX)
class Block(eqx.Module):
    feed_forward:       SwiGLUMLP   # frozen  — world knowledge
    feed_forward_prime: SwiGLUMLP   # adaptive — local context compression

# Forward:
h = layernorm(x)
x = x + feed_forward(h) + feed_forward_prime(h)
#           ^^^^^^^^^^^     ^^^^^^^^^^^^^^^^^^^^
#           frozen output   adapts during TTT
```

The inner optimizer spec from the 125M config:
```yaml
spec_inner: ["language_model.**.suffix_blocks.feed_forward_prime.**"]
optimizer_inner:
  optimizer_type: sgd
  lr: 1
  clip_gradient: 1.0
mini_batch_size: 1024
```

In [ ]:
# Parameter breakdown across model sizes
configs = [
    dict(label='Tiny (demo)',  vocab_size=64,  d_model=32,  n_layers=4,  d_ff=64),
    dict(label='Small',        vocab_size=512, d_model=128, n_layers=6,  d_ff=256),
    dict(label='Medium',       vocab_size=512, d_model=256, n_layers=8,  d_ff=512),
]

print(f"{'Config':<20} {'Total params':>14} {'TTT params':>12} {'TTT %':>8}")
print('-' * 58)
for cfg in configs:
    label = cfg.pop('label')
    m = E2ETTT(**cfg, n_heads=4, max_seq_len=512, window_size=32)
    total = m.count_params()
    ttt   = m.count_ttt_params()
    print(f"{label:<20} {total:>14,} {ttt:>12,} {100*ttt/total:>7.1f}%")
    print(f"  TTT block indices: {m.ttt_block_indices}")

---
## 6. Meta-Training

Meta-training (the **outer loop**) trains the model so the adaptive MLP starts from a good initialisation — one that adapts quickly with few gradient steps at test time.

```
For each sequence in the training set:
  ┌─ INNER LOOP (TTT — same objective used at test time) ──────────────────┐
  │  Split sequence into mini-batches of 1024 tokens                       │
  │  For each mini-batch:                                                   │
  │    1. Forward through prefix blocks (frozen SWA)                        │
  │    2. Forward through suffix blocks (adaptive MLP)                      │
  │    3. Compute next-token prediction loss                                │
  │    4. SGD step on adaptive MLP weights only                             │
  └────────────────────────────────────────────────────────────────────────┘
  OUTER LOOP: Backprop through the entire inner loop → update ALL weights
              (outer optimizer = AdamW)
```

In the JAX code this is `eqx.filter_value_and_grad` through `MetaModel.loss_for_sequence`.

In [ ]:
np.random.seed(42)
model = E2ETTT(
    vocab_size=64, d_model=32, n_heads=4, n_layers=4,
    d_ff=64, max_seq_len=256, window_size=16,
    ttt_batch_size=16, ttt_lr=0.005,
)
print(f"Model: {model.count_params():,} total params")
print(f"       {model.count_ttt_params():,} TTT (adaptive) params")
print(f"TTT block indices: {model.ttt_block_indices}")

In [ ]:
# Track meta-training loss
print("Running meta-training (200 steps)...")
t0 = time.time()
meta_losses = meta_train(model, n_steps=200, seq_len=64, lr=1e-3, verbose=True)
print(f"Done in {time.time()-t0:.1f}s")

fig, ax = plt.subplots(figsize=(9, 4))
w = 10
smoothed = np.convolve(meta_losses, np.ones(w)/w, mode='valid')
ax.plot(meta_losses, alpha=0.3, color='steelblue', label='Raw loss')
ax.plot(range(w-1, len(meta_losses)), smoothed,
        color='steelblue', linewidth=2, label=f'Smoothed (window={w})')
ax.set_xlabel('Meta-training step')
ax.set_ylabel('Cross-entropy loss')
ax.set_title('Meta-Training Loss Curve', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('meta_training_curve.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 7. Test-Time Training

At inference, when the model receives a long context document:

1. Process context in mini-batches
2. Each batch: forward → next-token loss → backward → **SGD on adaptive MLP only**
3. Adapted weights now encode a compressed representation of the context
4. Use adapted model for prediction (O(W) attention, not O(n²))
5. **Restore W₀** for the next document

In [ ]:
pattern = np.array([7, 13, 42, 3, 19, 55, 7, 13], dtype=np.int32)
context = np.tile(pattern, 20)   # 160 tokens

print(f"Context length : {len(context)} tokens")
print(f"Pattern        : {pattern.tolist()} (repeating)")

init_state = model.save_ttt_state()

logits_before = model.forward(context[:-1])
loss_before, _ = cross_entropy(logits_before, context[1:])

all_ttt_losses = []
for _ in range(5):
    all_ttt_losses.extend(model.ttt_adapt(context))

logits_after = model.forward(context[:-1])
loss_after, _ = cross_entropy(logits_after, context[1:])

print(f"\nLoss before TTT : {loss_before:.4f}")
print(f"Loss after TTT  : {loss_after:.4f}")
print(f"Improvement     : {100*(loss_before-loss_after)/loss_before:.1f}%")

---
## 8. Live Demo

Full end-to-end workflow: build model → meta-train → TTT adapt → evaluate.

In [ ]:
def demo():
    print("=" * 60)
    print("  E2E-TTT Demo (Pure NumPy — no GPU required)")
    print("=" * 60)

    np.random.seed(42)

    # 1. Build model
    m = E2ETTT(
        vocab_size=64, d_model=32, n_heads=4, n_layers=4, d_ff=64,
        max_seq_len=256, window_size=16, ttt_batch_size=16, ttt_lr=0.005,
    )
    total_p = m.count_params()
    ttt_p   = m.count_ttt_params()
    print(f"\nModel: {total_p:,} params total, {ttt_p:,} TTT params ({100*ttt_p/total_p:.1f}%)")
    print(f"TTT block indices: {m.ttt_block_indices}\n")

    # 2. Meta-train
    t0 = time.time()
    meta_train(m, n_steps=300, seq_len=64, lr=1e-3, verbose=True)
    print(f"Meta-training time: {time.time()-t0:.1f}s\n")

    # 3. TTT on structured context
    pattern = np.array([7, 13, 42, 3, 19, 55, 7, 13], dtype=np.int32)
    context = np.tile(pattern, 24)   # 192 tokens
    print(f"Test-time training on {len(context)}-token context...")
    print(f"Pattern: {pattern.tolist()} (repeating)")

    init_state = m.save_ttt_state()

    lb = model.forward(context[:-1])
    loss_before, _ = cross_entropy(lb, context[1:])

    t0 = time.time()
    ttt_losses = []
    for _ in range(3):
        ttt_losses.extend(m.ttt_adapt(context))
    ttt_time = time.time() - t0

    la = m.forward(context[:-1])
    loss_after, _ = cross_entropy(la, context[1:])

    print(f"\nResults:")
    print(f"  Loss before TTT : {loss_before:.4f}")
    print(f"  Loss after TTT  : {loss_after:.4f}")
    print(f"  Improvement     : {100*(loss_before-loss_after)/loss_before:.1f}%")
    print(f"  TTT time        : {ttt_time:.2f}s")

    # 4. Prediction quality
    test_in = context[:8]
    logits  = m.forward(test_in)
    pred    = np.argmax(logits, axis=-1)
    exp     = context[1:9]
    correct = np.sum(pred == exp)
    print(f"\nPrediction check (after TTT):")
    print(f"  Input    : {test_in.tolist()}")
    print(f"  Expected : {exp.tolist()}")
    print(f"  Got      : {pred.tolist()}")
    print(f"  Accuracy : {correct}/{len(exp)}")

    # 5. Restore
    m.restore_ttt_state(init_state)
    lr = m.forward(context[:-1])
    loss_restored, _ = cross_entropy(lr, context[1:])
    print(f"\nAfter restoring W₀: loss = {loss_restored:.4f}")
    print(f"(Matches pre-TTT: {abs(loss_restored - loss_before) < 1e-4})")

    print()
    print("=" * 60)
    print("  Summary: E2E-TTT Workflow")
    print("=" * 60)
    print("  1. Meta-train → good W₀ (adaptive MLP init)")
    print("  2. At test time: receive long context")
    print("  3. TTT: mini-batch SGD on next-token loss")
    print("     → context compressed into adaptive MLP weights")
    print("  4. Use adapted model for prediction (O(1) latency)")
    print("  5. Restore W₀ for next document")
    print("  Key: O(1) inference latency vs O(n²) for full attention")

demo()

---
## 9. Visualising TTT Adaptation

In [ ]:
# TTT loss curves across different pattern complexities
np.random.seed(0)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

test_patterns = [
    ('Short pattern (len=3)',  np.array([1, 5, 9], dtype=np.int32)),
    ('Medium pattern (len=6)', np.array([7, 13, 42, 3, 19, 55], dtype=np.int32)),
    ('Long pattern (len=10)',  np.array([2, 17, 4, 63, 9, 31, 7, 44, 11, 58], dtype=np.int32)),
]

for ax, (name, pattern) in zip(axes, test_patterns):
    context = np.tile(pattern, 30)[:128]

    m = E2ETTT(vocab_size=64, d_model=32, n_heads=4, n_layers=4,
               d_ff=64, max_seq_len=256, window_size=16,
               ttt_batch_size=16, ttt_lr=0.01)
    meta_train(m, n_steps=100, seq_len=48, lr=1e-3)

    ttt_losses_exp = []
    for _ in range(6):
        ttt_losses_exp.extend(m.ttt_adapt(context))

    ax.plot(ttt_losses_exp, color='#e67e22', linewidth=2)
    ax.fill_between(range(len(ttt_losses_exp)), ttt_losses_exp,
                    alpha=0.15, color='#e67e22')
    ax.set_title(name, fontweight='bold', fontsize=10)
    ax.set_xlabel('TTT step')
    if ax is axes[0]:
        ax.set_ylabel('NTP loss')
    ax.set_ylim(bottom=0)

plt.suptitle('TTT Loss Curves for Different Pattern Complexities',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('ttt_adaptation_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# TTT vs no-TTT prediction accuracy
np.random.seed(42)

m = E2ETTT(vocab_size=64, d_model=32, n_heads=4, n_layers=4,
           d_ff=64, max_seq_len=256, window_size=16,
           ttt_batch_size=16, ttt_lr=0.005)
meta_train(m, n_steps=200, seq_len=64, lr=1e-3)

pattern = np.array([7, 13, 42, 3, 19, 55, 7, 13], dtype=np.int32)
context = np.tile(pattern, 24)  # 192 tokens
targets = context[1:]

# Without TTT
preds_no_ttt = np.argmax(m.forward(context[:-1]), axis=-1)

# With TTT
init_state = m.save_ttt_state()
for _ in range(3):
    m.ttt_adapt(context)
preds_ttt = np.argmax(m.forward(context[:-1]), axis=-1)
m.restore_ttt_state(init_state)

# Cumulative accuracy
acc_no_ttt = np.cumsum(preds_no_ttt == targets) / np.arange(1, len(targets)+1)
acc_ttt    = np.cumsum(preds_ttt    == targets) / np.arange(1, len(targets)+1)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ax = axes[0]
ax.plot(acc_no_ttt, label='No TTT',   color='steelblue', linewidth=1.5)
ax.plot(acc_ttt,    label='With TTT', color='#e67e22',   linewidth=2)
ax.set_xlabel('Token position')
ax.set_ylabel('Cumulative accuracy')
ax.set_title('Cumulative Token Prediction Accuracy', fontweight='bold')
ax.legend()
ax.set_ylim(0, 1)

ax = axes[1]
N = 48
x = np.arange(N)
ax.bar(x - 0.2, (preds_no_ttt[:N] == targets[:N]).astype(float),
       width=0.35, label='No TTT',   color='steelblue', alpha=0.7)
ax.bar(x + 0.2, (preds_ttt[:N]    == targets[:N]).astype(float),
       width=0.35, label='With TTT', color='#e67e22',   alpha=0.9)
ax.set_xlabel('Token position (first 48)')
ax.set_ylabel('Correct (1=yes)')
ax.set_title('Per-Token Correctness (first 48 tokens)', fontweight='bold')
ax.legend()

plt.tight_layout()
plt.savefig('ttt_accuracy_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Overall accuracy — No TTT: {np.mean(preds_no_ttt==targets):.1%}  "
      f"| With TTT: {np.mean(preds_ttt==targets):.1%}")

---
## 10. Production JAX Implementation

The `test-time-training/e2e` repo is the full production implementation in JAX.

### Repository layout

```
e2e/
├── ttt/
│   ├── model/
│   │   ├── transformer.py   # SwiGLUMLP, Block, BlockCollectionSplit, MetaModel, CausalLM
│   │   ├── attention.py     # SWA (sliding-window attention) with RoPE
│   │   ├── loop.py          # train_on_sequence, Evaluator
│   │   └── loss.py          # cross_entropy_loss_and_accuracy
│   ├── config.py            # Hydra config dataclasses
│   ├── train.py             # Main training entry point
│   └── optimizers.py        # make_optimizer (SGD inner, AdamW outer)
└── configs/
    ├── model/               # 125m / 350m / 760m / 1b / 3b
    ├── experiment/          # pretrain + extension configs
    └── deploy/              # interactive (single node) + submitit (Slurm)
```

### Concept → production code mapping

| This notebook | JAX repo |
|---|---|
| `MLPBlock` | `SwiGLUMLP` (3-matrix SwiGLU) |
| `frozen_mlp` | `Block.feed_forward` |
| `adaptive_mlp` | `Block.feed_forward_prime` |
| `SlidingWindowAttention` | `SWA` (+ RoPE θ=500000, FlashAttention) |
| `LayerNorm` | `RMSNorm` (pre + post norm) |
| `ttt_adapt` mini-batch loop | `MetaModel.inner_loop_step` |
| `meta_train` outer loop | `train_on_sequence` + `filter_value_and_grad` |
| `save/restore_ttt_state` | `clone_pytree` + param restore in `loss_for_sequence` |
| `ttt_block_indices` | `suffix_len=3` in config |

### `MetaModel.loss_for_sequence` — pseudocode

```python
def loss_for_sequence(self, seq, state):
    # Split model into prefix (frozen SWA) and suffix (adaptive MLP) blocks
    prefix_blocks, suffix_blocks = split_at_suffix_len(blocks, suffix_len=3)

    # 1. Run prefix blocks ONCE — no TTT here
    prefix_output = prefix_call(prefix_blocks, embed(seq))

    # 2. Inner loop over mini-batches through suffix blocks
    for chunk in chunks(seq, size=1024):
        loss = suffix_call(suffix_blocks, prefix_output[chunk])
        grads = grad(loss, wrt=feed_forward_prime)
        suffix_blocks = sgd_update(suffix_blocks, grads, lr=1.0)

    # 3. Outer loss = avg loss across chunks
    return average_chunk_loss
```


---
## 11. Checkpoints & Reproducing Results

### Available checkpoints (Google Cloud Storage)

| Model | Dataset | Context | Bucket |
|---|---|---|---|
| 125M TTT-E2E (1× Chinchilla) | DCLM | 8K | `gs://ttt-e2e-checkpoints/125m_ttt_e2e_pretrain_dclm_8k_1x_cc` |
| 1B TTT-E2E (1× Chinchilla)   | DCLM | 8K | `gs://ttt-e2e-checkpoints/1b_ttt_e2e_pretrain_dclm_8k_1x_cc` |
| 3B TTT-E2E (3× Chinchilla)   | DCLM | 8K | `gs://ttt-e2e-checkpoints/3b_ttt_e2e_pretrain_dclm_8k_3x_cc` |
| 125M TTT-E2E (1× Chinchilla) | Books | 8K | `gs://ttt-e2e-checkpoints/125m_ttt_e2e_finetune_books_8k_1x_cc` |
| 1B TTT-E2E (1× Chinchilla)   | Books | 8K | `gs://ttt-e2e-checkpoints/1b_ttt_e2e_finetune_books_8k_1x_cc` |
| 3B TTT-E2E (3× Chinchilla)   | Books | 8K | `gs://ttt-e2e-checkpoints/3b_ttt_e2e_finetune_books_8k_3x_cc` |
| 3B TTT-E2E (3× Chinchilla)   | Books | **128K** | `gs://ttt-e2e-checkpoints/3b_ttt_e2e_finetune_books_128k_3x_cc` |

> **Note:** Requester Pays enabled — you need a GCP billing project.

### Setup & run

```bash
# 1. Install uv
curl -LsSf https://astral.sh/uv/install.sh | sh

# 2. Download tokenised dataset
gcloud storage cp -r gs://llama3-dclm-filter-8k/ llama3-dclm-filter-8k

# 3. Fill in deploy_paths in configs/deploy/interactive.yaml

# 4. Pretrain 125M on a single interactive node
uv run --exact train \
  +deploy=interactive \
  +experiment=125m/pretrain/pretrain-125m-e2e \
  training.wandb_entity=MY_ENTITY \
  training.wandb_project=MY_PROJECT \
  training.wandb_key=MY_KEY

# 5. Extend context to 32K
uv run --exact train \
  +deploy=interactive \
  +experiment=125m/extension/ext-125m-e2e-32K \
  training.resume_exp_name=pretrain-125m-e2e \
  training.load_part=params \
  training.wandb_entity=MY_ENTITY \
  training.wandb_project=MY_PROJECT \
  training.wandb_key=MY_KEY
```

### System requirements

- CUDA 12.8.1 · cuDNN 9.8.0 · NCCL 2.26.2
- JAX 0.4.x + Equinox + Optax
- Weights & Biases account

---
## Summary

| Concept | What it does | Where in code |
|---|---|---|
| **Sliding-Window Attention** | O(W) per-token; limits context to last W tokens | `ttt/model/attention.py → SWA` |
| **Adaptive MLP (prime)** | Compressed memory — updated by TTT | `feed_forward_prime` in `Block` |
| **Frozen MLP** | World knowledge from pre-training | `feed_forward` in `Block` |
| **Prefix blocks** | First N-3 layers — run once, no TTT | `BlockCollectionSplit.prefix_call` |
| **Suffix blocks** | Last 3 layers — carry adaptive MLP | `BlockCollectionSplit.suffix_call` |
| **Inner loop** | SGD on adaptive MLP (lr=1, per 1024-token chunk) | `MetaModel.inner_loop_step` |
| **Outer loop** | AdamW on ALL weights via BPTT through inner loop | `train_on_sequence` |
| **Weight restore** | Reset adaptive MLP to W₀ between documents | `save/restore_ttt_state` |

---
*Paper: [test-time-training.github.io/e2e.pdf](https://test-time-training.github.io/e2e.pdf)*  
*Code: [github.com/test-time-training/e2e](https://github.com/test-time-training/e2e)*
